# Set up SQLDoom on Lakebase in this workspace

Runs `setup_workspace.py` (next to this notebook), step by step. Every step checks what is already there, so you can run the notebook again after a failure.

**Before you start**
* Upload the shareware `DOOM1.WAD` to a Unity Catalog volume and put its path in the `wad` widget.
* For the music (single player): upload a General MIDI soundfont next to it, `TimGM6mb.sf2` (GPL-2; in Debian's package `timgm6mb-soundfont`, `/usr/share/sounds/sf2/TimGM6mb.sf2`), or put another `.sf2` path in the `soundfont` widget. Without one the game plays without music; you can add it later and run `s.run(["music"])`.
* Run on serverless compute (or any compute with network access to Lakebase). The notebook runs as you: you become the owner of the Lakebase project.

**What it does**
1. Cell 3, database part (about 1 minute): Lakebase project (Postgres 19, default endpoint size 16 CU), schema `doom` owned by role `doom_owner`, the tables and the game's SQL from `sql/`, then the game's definitions and the 9 maps from `DOOM1.WAD`, and the music rendered with the soundfont.
2. Cell 4, app part: the Databricks App with its Lakebase resource, its service principal granted `doom_owner` and **Can manage** on the project, then branch `game-template` (after the app, so it copies the app's database login), `app.yaml` pointed at your project, the lobby endpoint sized down to 1 CU, and a final check.
3. Cell 5, optional: deploy the app from this repository's folder. To deploy from Git instead, commit `app.yaml` and deploy from the app page.

Afterwards: give players **Can use** on the app. Details and troubleshooting: `docs/DEPLOYMENT.md`.

In [0]:
%pip install -q "databricks-sdk>=0.118.0" "psycopg2-binary>=2.9" soundfile numpy
# the music's renderer (sqldoom.wad.midi); --no-deps: its pyaudio dependency, for live playback
# only, needs PortAudio to build
%pip install -q --no-deps tinysoundfont
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("project", "sqldoom", "Lakebase project id")
dbutils.widgets.text("app", "sqldoom-ltap", "App name")
dbutils.widgets.text("wad", "/Volumes/<catalog>/<schema>/<volume>/DOOM1.WAD", "DOOM1.WAD path")
dbutils.widgets.text("soundfont", "", "Music soundfont (.sf2; empty: TimGM6mb.sf2 next to the WAD)")

import os, sys
here = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
setup_dir = "/Workspace" + here if not here.startswith("/Workspace") else here   # <repo>/setup
sys.path[:0] = [setup_dir]                                                            # setup_workspace adds <repo>/src

from setup_workspace import Setup
s = Setup(project=dbutils.widgets.get("project"), app=dbutils.widgets.get("app"), wad=dbutils.widgets.get("wad"),
          soundfont=dbutils.widgets.get("soundfont") or None)
s.run(until="sql")   # game-template comes after the app (cell 4): it must copy the app's login

In [0]:
s.run(start="app")
print("everything in place:", s.status())

In [0]:
# Deploys the repository's folder (the parent of setup/) as it is in the workspace, app.yaml included.
# The first start installs the current SQL on game-template (about 25 s); then open the app URL.
s.deploy()
print(s.w.apps.get(name=s.app_name).url)